# **Modelos NLP**

In [1]:
# CONFIGURACIÓN INICIAL
# =========================

import os, time, ast, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)

from transformers import (
    get_scheduler,
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)


In [ ]:
# MODULO DE MANEJO DE DATOS
# ---------------------------------

#  [Carga de datos]
df = pd.read_json('../Data/clean_words.json', lines=True)
df = df.rename(columns={'macro_label': 'Category'})
print(f"Registros totales: {len(df)}")

# Codificar etiquetas
le = LabelEncoder()
df["label"] = le.fit_transform(df["Category"])

# Dividir dataset
train, temp = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df["label"])
val, test = train_test_split(temp, test_size=0.5, random_state=SEED, stratify=temp["label"])

print(f"Distribución de los datos: \nTrain: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Registros totales: 2483


## DistilBERT

DistilBERT es un modelo de lenguaje basado en la arquitectura Transformer, entrenado mediante knowledge distillation a partir de BERT.

Recibe:
    * Una secuencia de tokens numéricos que representan el texto. 
    * Una máscara binaria de tokens atentidos e ignorados
    * Labels de clasificación

Estas entradas se generan usando el tokenizador _DistilBertTokenizerFast_ , que convierte texto en tensores compatibles con el modelo.


In [ ]:
# Tokenización y codificación de etiquetas
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Tokenización
def tokenize_texts(texts):
    return tokenizer(
        [' '.join(x) for x in texts],
        padding=True,
        truncation=True,
        return_tensors='pt'
    )

train_texts = train_texts.sample(frac=0.5, random_state=42)
train_labels = train_labels.loc[train_texts.index]

val_texts = val_texts.sample(frac=0.5, random_state=42)
val_labels = val_labels.loc[val_texts.index]

test_texts = test_texts.sample(frac=0.5, random_state=42)
test_labels = test_labels.loc[test_texts.index]

train_encodings = tokenize_texts(train_texts)
val_encodings = tokenize_texts(val_texts)
test_encodings = tokenize_texts(test_texts)

In [4]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)

num_labels = df['label_id'].nunique()

# Modelo
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',  # o distilbert-base-uncased si es en inglés
    num_labels=num_labels
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Función de métricas personalizada
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    prec = precision_score(labels, preds, average='weighted')
    rec = recall_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec}

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


In [ ]:
# Entrenamiento
start_time = time.time()
trainer.train()
end_time = time.time()
train_time = end_time - start_time
print(f"\n Tiempo total de entrenamiento: {train_time:.2f} segundos")

save_path = "./distilbert_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Modelo guardado en: {save_path}")

c:\Users\danie\anaconda3\envs\NLP\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
# Validación
val_metrics = trainer.evaluate(val_dataset)
val_acc = val_metrics.get('eval_accuracy', None)
print(f"\Accuracy de validación: {val_acc:.4f}")

# Prueba
test_metrics = trainer.evaluate(test_dataset)
test_acc = test_metrics.get('eval_accuracy', None)
print(f" Accuracy de prueba: {test_acc:.4f}")

# 4. Predicciones y reporte de clasificación
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

# Reporte
report = classification_report(
    true_labels,
    preds,
    target_names=le.classes_,
    digits=4
)

print("\n Classification Report:\n")
print(report)
report_path = "./classification_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("DistilBERT Classification Report\n")
    f.write(f"Tiempo de entrenamiento: {train_time:.2f} segundos\n")
    f.write(f"Accuracy validación: {val_acc:.4f}\n")
    f.write(f"Accuracy prueba: {test_acc:.4f}\n\n")
    f.write(report)

print(f" Reporte guardado en: {report_path}")